<a href="https://colab.research.google.com/github/Divyankseervi/crash-prediction/blob/main/Crash_Prediction_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/Divyankseervi/crash-prediction.git
%cd crash-prediction

Cloning into 'crash-prediction'...
remote: Enumerating objects: 48, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 48 (delta 15), reused 47 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (48/48), 2.73 MiB | 10.45 MiB/s, done.
Resolving deltas: 100% (15/15), done.
/content/crash-prediction


In [2]:
!ls -R

.:
backend				      frontend
crash_severity_analysis_v2.png	      requirements.txt
crash_severity_enhanced_analysis.png

./backend:
AV_accident_data__1_.xlsx  main.py	       preprocess.py
__init__.py		   model_results.json  static

./backend/static:
rf_model.joblib  scaler.joblib

./frontend:
analysis.html	dataset.html  predict.html
dashboard.html	index.html    style.css


In [3]:
!pip install fastapi uvicorn numpy pandas scikit-learn scipy matplotlib seaborn imbalanced-learn openpyxl joblib

In [5]:
import joblib

model = joblib.load("backend/static/rf_model.joblib")
scaler = joblib.load("backend/static/scaler.joblib")

print("Model loaded successfully!")
print("Model:", type(model))
print("Number of trees:", model.n_estimators)
print("Classes:", model.classes_)

Model loaded successfully!
Model: <class 'sklearn.ensemble._forest.RandomForestClassifier'>
Number of trees: 500
Classes: [0 1 2 3]


In [6]:
import numpy as np
import pandas as pd
import json

# Load feature names
with open("backend/model_results.json", "r") as f:
    results = json.load(f)

feat_names = results["dataset"]["feature_names"]

# Sample crash conditions
speed_limit = 35
mileage = 500
precrash_speed = 50
is_night = 1
is_wet = 0
is_highway = 0
is_dark = 1
is_bad_weather = 0
airbag = 1

# Derived features
speed_ratio = precrash_speed / speed_limit
speed_night = precrash_speed * is_night

# Scale numeric features
scaled = scaler.transform([[
    speed_limit,
    mileage,
    precrash_speed,
    speed_ratio,
    speed_night
]])[0]

# Create complete feature row
row = {col: 0.0 for col in feat_names}

row["Posted Speed Limit (MPH)"] = scaled[0]
row["Mileage"] = scaled[1]
row["SV Precrash Speed (MPH)"] = scaled[2]
row["Speed_Ratio"] = scaled[3]
row["Speed_Night"] = scaled[4]

row["Is_Night"] = is_night
row["Is_Wet"] = is_wet
row["Is_Highway"] = is_highway
row["Is_Dark"] = is_dark
row["Is_BadWeather"] = is_bad_weather
row["AirBag_Deployed"] = airbag
row["Incident_Year"] = 2024
row["Incident_Month"] = 6
row["Is_OldVehicle"] = 0

# Convert to DataFrame in correct feature order
X_test_input = pd.DataFrame([row])[feat_names].astype(float)

# Predict
prediction = model.predict(X_test_input)[0]
probabilities = model.predict_proba(X_test_input)[0]

labels = ["POD", "Minor", "Moderate", "Serious"]

print("===== CRASH SEVERITY PREDICTION =====")
print("Predicted Severity:", labels[prediction])
print("\nProbabilities:")

for label, prob in zip(model.classes_, probabilities):
    print(f"{labels[label]}: {prob:.2%}")

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


===== CRASH SEVERITY PREDICTION =====
Predicted Severity: POD

Probabilities:
POD: 84.90%
Minor: 11.05%
Moderate: 0.45%
Serious: 3.60%
